# Vòng private: chạy ĐÚNG giải pháp đã nộp trên `private_test` (625 mẫu) và lấy `submission.jsonl` để nộp. Khoảng **1 đến 1,5 giờ GPU**.
Giống hệt A7a, chỉ đổi split: ASR sống (`phowl_ft/merged`) + Qwen3-4B đã gộp LoRA (`q4b_merged`) + luật, offline, đường mã `run_container.py` của image. **Không đổi gì so với bài đã nộp** (commit `submission-v1` = `98d3199`, model sha256 `652e96ae…` và `cffba381…`); cell R-1 kiểm cả mã lẫn trọng số, lệch là DỪNG. Hướng dẫn: `docs/HUONG_DAN_PRIVATE.md`.

**Trước khi chạy:** notebook **Private**, Internet **Off**, Accelerator **GPU RTX PRO 6000** (nếu không có thì L4; ghi lại GPU đã dùng).
**Input cần gắn (3 thứ):**
1. dataset `lifemate-code` (bản nào cũng được nếu mã suy luận trùng `98d3199`; R-1 kiểm từng file)
2. dataset **Private** mới `lifemate-private` chứa thư mục `private_test/` (giải nén từ `private_test.zip` của BTC: `private_test.jsonl`, `private_test_tools_catalog.json`, `private_test_environment/`, `audio/`, `zaic_tools/`). **Bắt buộc Private**, BTC cấm phát tán.
3. **output của notebook `merge-lora`** (có `checkpoints/q4b_merged`, `checkpoints/phowl_ft/merged`)

**KHÔNG gắn** `Hello_done`, `a8-full` hay output khác có `checkpoints/` hoặc `state/`.
**Thứ tự cell:** P-1 đến P-3, R-1 (kiểm mã, trọng số, dữ liệu), R-2 (thử 20 mẫu × 2 lần), R-3 (đầy đủ 625 mẫu × 2 lần), R-4 (tách file nộp, đóng zip), R-5 (dọn, chạy cuối rồi Save Version).
**Gửi Phương qua Drive riêng (không vào git, không công khai):** `private_results.zip` (có `private_submission.jsonl` là file sẽ nộp, và `report.json`). Lỗi ở bất kỳ cell nào: dừng, gửi **nguyên văn**.

In [ ]:
%%bash
# P-1: setup. Dùng kaggle_setup.sh của Phương; chỉ đặt đường dẫn model cho khớp đúng thứ ta gắn:
#  - QWEN = đường dẫn giả (không dùng Qwen3-4B gốc ở notebook này)
#  - PHO = đường dẫn giả (A5 không dùng PhoWhisper)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
QW=/nonexistent/qwen3-4b   # bước này không dùng Qwen3-4B gốc (LLM là model đã gộp trong checkpoints/)
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e 's#^PHO=.*#PHO=/nonexistent/phowhisper#' -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục từ output notebook trước? (checkpoints và outputs)"; ls /kaggle/working/checkpoints | head -20; ls /kaggle/working/outputs | head -12

In [ ]:
%%bash
# P-2: in phiên bản torch/CUDA/GPU, kiểm tra GPU, peft, accelerate và việc transformers nhận qwen3_5
pip uninstall -y torchao 2>&1 | tail -1
python -X utf8 - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers nhận qwen3:", "qwen3" in CONFIG_MAPPING, "| qwen3_5 (Qwen3.5):", "qwen3_5" in CONFIG_MAPPING)
try:
    import accelerate
    print("accelerate", accelerate.__version__)
except ImportError:
    print("THIẾU accelerate: gen_text.py sẽ lỗi ở bước nạp model (device_map). Báo lại, đừng pip install từ mạng.")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại")
print("VRAM tổng: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
EOF

In [ ]:
%%bash
# P-3: ghi sẵn đoạn mã đóng gói (chỉ file kết quả nhỏ, không có nội dung dữ liệu)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# R-1: kiểm (a) mã suy luận trùng commit nộp 98d3199 từng file, (b) trọng số trùng models.lock.json của image đã nộp, (c) dữ liệu private đủ. Lệch là DỪNG.
# Không mở private_test_environment/ (chỉ zaic_tools đọc nó khi chạy).
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
cat > /kaggle/working/want_code.json <<'JSONEOF'
{
"scripts/run_container.py": "5518922c170d5a2b1ca6333db7d84bc321557b2b8fb8c5879f165685af887903",
"scripts/verify_image.py": "c0d9974b6130331cfaf053c6ec16245c0fba6190cb189337302e02ebcd56f23a",
"scripts/verify_submission.py": "471ea517dca506212ed5d459d879133667853f38e57c5231f339c868a28d348a",
"src/lifemate/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
"src/lifemate/agent.py": "b8b868a7f719f807154bd08ac2cc49dbf8fc97239e3344ff67b348b9a97edbc8",
"src/lifemate/asr.py": "3ab0d25170998c38d44ea2c6a3bcf96864b4899a233ce0bc3876a5fa02c7fc56",
"src/lifemate/augment.py": "c6dbf4c801d658641599f7848d4bffe1286ad927cf321127831efb50dd032e91",
"src/lifemate/callgen.py": "3d16d87296afc693a7b19e316e6930991e4da14f1e083db436988fdd0f059b5f",
"src/lifemate/container.py": "076009875b79f09c7295a9fce7b95fdb471b3204aa94cd536053f2f7025cb261",
"src/lifemate/datacheck.py": "0dc7d319fd92614cf800b3857df68bbd4f8f38f83a5b1f08fe7515329dd2abe6",
"src/lifemate/determinism.py": "596e318ca4821bc561e33a7792c1e35e00efbacdd364aacb7de654dbd775be2d",
"src/lifemate/grouping.py": "86ede6a00d7e09307daa601c66eeecc1b351b7dc31a43b608172d1cdcb32faab",
"src/lifemate/jsonl.py": "6f279278ca2d2690b08d22ac7f44ae6d34eada1a004934febb049142a578a6a1",
"src/lifemate/leakcheck.py": "915716f17f97e00cb15a8644889a7f499260cdcc7e52643156e8a5ab512ecdb0",
"src/lifemate/manifest.py": "91e7e8643dcd1bbb9adb88e4fb49d1331113d012f3cbb8eb7af60ff4c0d6fbbe",
"src/lifemate/masking.py": "1fa47addc3c28cec893dac1ded090479155c5437782742d5112ececa4d1242f1",
"src/lifemate/metric.py": "d09f594293c207400f2a26cec1e8f002ddeab8ed7607b210be9dc1bef10d10cd",
"src/lifemate/normalize.py": "209d7138cdb78d3b4a672d0c95e4f188c983fb469b5468b676c2d1c7cb003e4f",
"src/lifemate/params.py": "b22b3dccbe09fed91f0e0fd40c2aa384dc8eeab790ffbbd7049e704abfe6662a",
"src/lifemate/paths.py": "31e9a8d88cfbbb75e69db8d64a999bfa270286a6ef35e9fc7f02ea6746bf29dc",
"src/lifemate/pipeline.py": "8a9ae31922a0f91a0c71cd8de15844b3a7bf4baf584a0ea904f1da900f7ce351",
"src/lifemate/prompt.py": "0516e239273fc5e89560514f66f722900d4a93f8825977d278c32786c18eb791",
"src/lifemate/rules.py": "97999473bdcaafc505d27ab61f6561d29ab8a3a2658264a3b608aaa583c1a060",
"submission/predict.py": "72f780273f64b21f3ee0c8fe1882374a55831667a88c095991c345fa501a506a"
}
JSONEOF
python -X utf8 - <<'EOF' || MISSING=1
import hashlib, json, sys
from pathlib import Path
want = json.load(open("/kaggle/working/want_code.json", encoding="utf-8"))
bad = []
for f, h in want.items():
    p = Path(f)
    got = hashlib.sha256(p.read_bytes().replace(b"\r\n", b"\n")).hexdigest() if p.is_file() else "THIẾU"
    if got != h:
        bad.append(f"{f}: {'THIẾU' if got == 'THIẾU' else 'KHÁC'}")
print(f"mã suy luận: {len(want) - len(bad)}/{len(want)} file trùng commit nộp 98d3199")
for b in bad:
    print("   ", b)
sys.exit(1 if bad else 0)
EOF
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need checkpoints/q4b_merged/model.safetensors
need checkpoints/q4b_merged/tokenizer_config.json
need checkpoints/phowl_ft/merged/model.safetensors
python -X utf8 - <<'EOF' || MISSING=1
import hashlib, sys
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(1 << 24):
            h.update(b)
    return h.hexdigest()
ok = True
for name, path, want in (("llm", "checkpoints/q4b_merged/model.safetensors", "652e96aed23e77efcc1fdce04ef7586f322cf36499fb2b958d6e10e2086c5882"),
                         ("asr", "checkpoints/phowl_ft/merged/model.safetensors", "cffba381278e485f69e14888536e610ab415727dc1bde031b17097a0eff64d2c")):
    got = sha(path)
    print(f"{name}: {'TRÙNG bài nộp' if got == want else 'KHÁC bài nộp: DỪNG'} ({got[:12]}…)")
    ok &= got == want
sys.exit(0 if ok else 1)
EOF
J=$(find /kaggle/input -maxdepth 7 -path "*private_test/private_test.jsonl" -print -quit 2>/dev/null)
if [ -z "$J" ]; then echo "THIẾU  private_test/private_test.jsonl trong input (dataset lifemate-private)"; MISSING=1; else
  P=$(dirname "$J"); DR=$(dirname "$P"); echo "export DR=$DR" > /kaggle/working/priv_env.sh; echo "data root: $DR"
  need "$P/private_test_tools_catalog.json"
  [ -d "$P/audio" ] && echo "ok     audio/" || { echo "THIẾU  audio/"; MISSING=1; }
  [ -f "$P/zaic_tools/__init__.py" ] && echo "ok     zaic_tools/" || { echo "THIẾU  zaic_tools/"; MISSING=1; }
  [ -d "$P/private_test_environment" ] && echo "ok     private_test_environment/ (có, không mở)" || { echo "THIẾU  private_test_environment/"; MISSING=1; }
  n=$(grep -c . "$J"); echo "private_test.jsonl: $n mẫu"; [ "$n" = 625 ] || { echo "phải là 625 mẫu"; MISSING=1; }
fi
if [ "$MISSING" = 1 ]; then echo "-> THIẾU hoặc LỆCH: dừng, gửi nguyên văn output cell này cho Phương."; exit 1; fi
echo "-- dọn bản sao không dùng, vá bản sao verify_image.py (bỏ -I khi gọi Python con, như A7a; mã trong repo không đổi)"
for d in checkpoints/*; do case "$(basename "$d")" in q4b_merged|phowl_ft) ;; *) rm -rf "$d" ;; esac; done
for d in checkpoints/phowl_ft/*; do [ "$(basename "$d")" = "merged" ] || rm -rf "$d"; done
rm -rf /kaggle/working/outputs/*; mkdir -p outputs/logs
sed -i 's/"-I", "-X", "utf8"/"-X", "utf8"/g' scripts/verify_image.py
df -h /kaggle/working | tail -1

In [ ]:
%%bash
# R-2 (khoảng 3 đến 6 phút): thử nhanh 20 mẫu × 2 lần để biết đường chạy thông trên private_test. Lỗi thì dừng, báo NGUYÊN VĂN.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/priv_env.sh
python -X utf8 scripts/verify_image.py --no-docker --data-root "$DR" --split private_test --runs 2 --limit 20 \
  --out-dir /kaggle/working/check_priv_quick -- --llm checkpoints/q4b_merged --adapter none --asr checkpoints/phowl_ft/merged 2>&1 | tail -n 25
echo "exit: ${PIPESTATUS[0]} (0 = ALL CHECKS PASSED)"

In [ ]:
%%bash
# R-3 (khoảng 50 đến 80 phút trên RTX PRO 6000): 625 mẫu × 2 lần, cùng đường mã của image; verify_submission phát lại hash; so hai lần; đo thời gian.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/priv_env.sh
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader | tee outputs/logs/priv_gpu.txt
START=$(date +%s)
python -X utf8 scripts/verify_image.py --no-docker --data-root "$DR" --split private_test --runs 2 \
  --out-dir /kaggle/working/check_priv -- --llm checkpoints/q4b_merged --adapter none --asr checkpoints/phowl_ft/merged 2>&1 | tee outputs/logs/priv_verify_image.txt | tail -n 30
echo "verify_image exit code: ${PIPESTATUS[0]} (0 = ALL CHECKS PASSED)"
echo "tổng thời gian: $(( $(date +%s) - START )) s" | tee -a outputs/logs/priv_verify_image.txt

In [ ]:
%%bash
# R-4: lấy file nộp = submission của LẦN 1, kiểm 625 dòng đúng id và định dạng, in số tổng hợp (không in nội dung), đóng zip.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/priv_env.sh
cp /kaggle/working/check_priv/run1/submission.jsonl /kaggle/working/private_submission.jsonl
python -X utf8 - <<'EOF' | tee outputs/logs/priv_summary.txt
import json, os
from collections import Counter
dr = os.environ.get("DR") or open("/kaggle/working/priv_env.sh").read().split("=", 1)[1].strip()
ids = [json.loads(l)["id"] for l in open(f"{dr}/private_test/private_test.jsonl", encoding="utf-8") if l.strip()]
rows = [json.loads(l) for l in open("/kaggle/working/private_submission.jsonl", encoding="utf-8") if l.strip()]
got = [r["id"] for r in rows]
print("dòng:", len(rows), "| id trùng tập private:", sorted(got) == sorted(ids), "| id lặp:", len(got) - len(set(got)))
print("trường:", Counter(tuple(sorted(r)) for r in rows))
print("tool_calls rỗng:", sum(not r["tool_calls"] for r in rows), "| call trung bình: %.2f | nhiều nhất: %d" % (sum(len(r["tool_calls"]) for r in rows) / len(rows), max(len(r["tool_calls"]) for r in rows)))
t = sorted(r.get("time", 0) for r in rows)
print("time ms: mean %.0f | p95 %d | max %d | mẫu > 20 s: %d | > 25 s: %d" % (sum(t) / len(t), t[int(.95 * len(t))], t[-1], sum(x > 20000 for x in t), sum(x > 25000 for x in t)))
EOF
python -X utf8 -c "
import json
r = json.load(open('/kaggle/working/check_priv/report.json', encoding='utf-8'))
print(json.dumps({k: v for k, v in r.items() if k != 'runs'}, ensure_ascii=False, indent=1))
for e in r['runs']: print(e)
" 2>&1 | cut -c1-300 | tee outputs/logs/priv_report_summary.txt
sha256sum /kaggle/working/private_submission.jsonl | tee outputs/logs/priv_submission_sha256.txt
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/private_results.zip \
  /kaggle/working/private_submission.jsonl /kaggle/working/check_priv/report.json /kaggle/working/check_priv_quick/report.json \
  outputs/logs/priv_*.txt /kaggle/working/torch_cuda.txt

In [ ]:
%%bash
# R-5 (CUỐI CÙNG): dọn (xóa lifemate/, trọng số), giữ check_priv/, private_submission.jsonl, zip và log. Rồi Save Version (Private).
cd /kaggle/working
rm -rf lifemate helpers setup_patched.sh config_paths.txt checkpoints state want_code.json
du -sh /kaggle/working/* 2>/dev/null
echo "xong. Save Version (private). Tải private_results.zip ở tab Output, gửi Phương qua Drive riêng."